In [2]:
%pip install numpy scikit-learn

  Using cached numpy-2.5.3-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached scikit_learn-1.9.1-cp314-cp314-win_amd64.whl.metadata (9.3 kB)
  Using cached scipy-1.18.1-cp314-cp314-win_amd64.whl.metadata (61 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached narwhals-2.26.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.7.0-py3-none-any.whl.metadata (24 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
Using cached numpy-2.5.3-cp314-cp314-win_amd64.whl (12.7 MB)
Using cached scikit_learn-1.9.1-cp314-cp314-win_amd64.whl (8.4 MB)
Using cached joblib-1.6.0-py3-none-any.whl (306 kB)
Using cached cloudpickle-3.1.2-py3-none-any.whl (22 kB)
Using cached narwhals-2.26.0-py3-none-any.whl (474 kB)
Using cached scipy-1.18.1-cp314-cp314-win_amd64.whl (37.4 MB)
Using cached threadpoolctl-3.7.0-py3-none-any.whl (26 kB)

   ----- ---------------------------------- 1/7 [numpy]
   ----- ---------------------------------- 1/7

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [3]:
import numpy as np
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=400, noise=0.20, random_state=1)
y = y.reshape(-1, 1).astype(float)
X = (X - X.mean(0)) / X.std(0)

def relu(z): return np.maximum(0, z)
def drelu(z): return (z > 0).astype(float)
def sig(z): return 1 / (1 + np.exp(-z))

def init(sizes, seed=0):
    rng = np.random.default_rng(seed); P = []
    for a, b in zip(sizes[:-1], sizes[1:]):
        W = rng.normal(0, np.sqrt(2/a), size=(a, b))
        P.append([W, np.zeros((1, b))])
    return P

def grads(P, Xb, yb):
    zs, acts, a = [], [Xb], Xb
    for i, (W, b) in enumerate(P):
        z = a @ W + b; zs.append(z)
        a = sig(z) if i == len(P)-1 else relu(z); acts.append(a)
    m = Xb.shape[0]; g = [None] * len(P)
    dz = (acts[-1] - yb) / m
    for i in reversed(range(len(P))):
        g[i] = [acts[i].T @ dz, dz.sum(0, keepdims=True)]
        if i > 0:
            dz = (dz @ P[i][0].T) * drelu(zs[i-1])
    return g
def accuracy(P, X, y):
    a = X
    for i, (W, b) in enumerate(P):
        a = sig(a @ W + b) if i == len(P)-1 else relu(a @ W + b)
    return np.mean((a > 0.5) == (y > 0.5))

sizes, lr, EPOCHS = [2, 16, 16, 1], 0.5, 200

P = init(sizes, seed=0)
for epoch in range(EPOCHS):
    g = grads(P, X, y)
    for k in range(len(P)):
        P[k][0] -= lr * g[k][0]
        P[k][1] -= lr * g[k][1]
print("Batch GD accuracy:", accuracy(P, X, y))

P = init(sizes, seed=0); B = 16; idx = np.arange(len(X))
for epoch in range(EPOCHS):
    np.random.shuffle(idx)
    for s in range(0, len(X), B):
        b = idx[s:s+B]
        g = grads(P, X[b], y[b])
        for k in range(len(P)):
            P[k][0] -= lr * g[k][0]
            P[k][1] -= lr * g[k][1]
print("SGD accuracy:", accuracy(P, X, y))

Batch GD accuracy: 0.965
SGD accuracy: 0.965


In [4]:
%pip install tensorflow

  Using cached tensorflow-2.22.0rc0-cp314-cp314-win_amd64.whl.metadata (4.6 kB)
  Using cached absl_py-2.5.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-7.36.2-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached wrapt-2.5.0-cp314-cp314-win_amd64.whl.metadata (7.7 kB)
  Using cached grpcio-1.84.0-cp314-cp314-win_amd64.whl.metadata (3.9 kB)
  Using cached keras_nightly-3.16.0.dev2026093004-py3-none-any.whl.metadata (6.3 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import SGD

model = Sequential([
    Input(shape=(2,)),
    Dense(16, activation='relu'),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer=SGD(learning_rate=0.5),
    metrics=['accuracy']
)

# Batch Gradient Descent → use the whole dataset as one batch
model.fit(
    X,
    y,
    epochs=200,
    batch_size=len(X),
    verbose=0
)

# Stochastic / mini-batch → small batch size, many updates per epoch
model.fit(
    X,
    y,
    epochs=200,
    batch_size=16,
    verbose=0
)